# Ejercicios 2 · Inserción, actualización, proyección y filtrado

Colección `terremotos.terremotos`: 19.217 terremotos de magnitud ≥ 5 registrados por USGS en todo el mundo desde 2016.

In [1]:
import os
from datetime import datetime, timezone

import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient, GEOSPHERE
from pymongo.errors import BulkWriteError

load_dotenv()
client = MongoClient(os.environ["MONGO_URI"], serverSelectionTimeoutMS=5000)
col = client["terremotos"]["terremotos"]

pd.set_option("display.max_colwidth", 60)

def tabla(cursor):
    """Muestra el resultado de un find() como tabla (los subdocumentos se aplanan: impacto.personas_lo_sintieron)."""
    return pd.json_normalize(list(cursor))

# Por si se re-ejecuta el notebook: elimina los documentos insertados a mano
col.delete_many({"_id": {"$regex": "^manual-"}})
print(col.count_documents({}), "documentos en la colección")

0 documentos en la colección


## 2.1 Inserción

### Ejercicio 1 — Insertar un documento (`insert_one`)
El dataset empieza en 2016, así que no incluye el terremoto de **Lorca (Murcia) de 2011**, el más dañino en España en décadas.
Lo insertamos a mano respetando la misma estructura que el resto de documentos (datos aproximados del IGN).

In [2]:
lorca = {
    "_id": "manual-lorca-2011",
    "magnitud": 5.1,
    "tipo_magnitud": "mw",
    "lugar": "Lorca, Murcia, Spain",
    "region": "Spain",
    "fecha": datetime(2011, 5, 11, 16, 47, 25, tzinfo=timezone.utc),
    "ubicacion": {"type": "Point", "coordinates": [-1.70, 37.72]},
    "profundidad_km": 2,
    "tipo_evento": "earthquake",
    "tsunami": False,
    "estado": "reviewed",
    "fuentes": ["manual"],
    "productos": [],
}
resultado = col.insert_one(lorca)
print("Insertado con _id:", resultado.inserted_id)
col.find_one({"_id": "manual-lorca-2011"})

Insertado con _id: manual-lorca-2011


{'_id': 'manual-lorca-2011',
 'magnitud': 5.1,
 'tipo_magnitud': 'mw',
 'lugar': 'Lorca, Murcia, Spain',
 'region': 'Spain',
 'fecha': datetime.datetime(2011, 5, 11, 16, 47, 25),
 'ubicacion': {'type': 'Point', 'coordinates': [-1.7, 37.72]},
 'profundidad_km': 2,
 'tipo_evento': 'earthquake',
 'tsunami': False,
 'estado': 'reviewed',
 'fuentes': ['manual'],
 'productos': []}

### Ejercicio 2 — Insertar varios documentos (`insert_many`) controlando duplicados
Insertamos dos documentos: el terremoto de **Santa Fe (Granada) de 2021** y, por error, una copia de un terremoto que ya existe (Kamchatka 2025, `_id: us6000qw60`).
Con `ordered=False` MongoDB inserta todos los documentos válidos y nos informa de los que fallan por `_id` duplicado.

In [3]:
nuevos = [
    {
        "_id": "manual-granada-2021",
        "magnitud": 4.4,
        "tipo_magnitud": "mw",
        "lugar": "Santa Fe, Granada, Spain",
        "region": "Spain",
        "fecha": datetime(2021, 1, 26, 20, 36, 0, tzinfo=timezone.utc),
        "ubicacion": {"type": "Point", "coordinates": [-3.71, 37.19]},
        "profundidad_km": 3,
        "tipo_evento": "earthquake",
        "tsunami": False,
        "estado": "reviewed",
        "fuentes": ["manual"],
        "productos": [],
    },
    {"_id": "us6000qw60", "magnitud": 8.8, "lugar": "Kamchatka (duplicado)"},
]
try:
    col.insert_many(nuevos, ordered=False)
except BulkWriteError as e:
    print("Insertados:", e.details["nInserted"])
    for error in e.details["writeErrors"]:
        print("Error", error["code"], "->", error["keyValue"])

print("Documentos manuales:", col.count_documents({"fuentes": "manual"}))

Insertados: 1
Error 11000 -> {'_id': 'us6000qw60'}
Documentos manuales: 2


## 2.2 Actualización

### Ejercicio 3 — Limpieza de datos (`update_many`)
El campo `estado` debería valer siempre `"reviewed"`, pero hay un documento con `"REVIEWED"` en mayúsculas. Lo normalizamos.

In [4]:
print("Antes:  ", col.distinct("estado"))
resultado = col.update_many({"estado": "REVIEWED"}, {"$set": {"estado": "reviewed"}})
print("Modificados:", resultado.modified_count)
print("Después:", col.distinct("estado"))

Antes:   ['reviewed']
Modificados: 0
Después: ['reviewed']


### Ejercicio 4 — Varios operadores a la vez (`update_one` con `$set`, `$addToSet` y `$currentDate`)
Completamos el documento de Lorca: añadimos el número de fallecidos dentro del subdocumento `impacto`,
añadimos el IGN como fuente (sin duplicarla si ya estuviera) y registramos la fecha de la modificación.

In [5]:
resultado = col.update_one(
    {"_id": "manual-lorca-2011"},
    {
        "$set": {"impacto.fallecidos": 9, "alerta": "orange"},
        "$addToSet": {"fuentes": "ign"},
        "$currentDate": {"modificado": True},
    },
)
print("Modificados:", resultado.modified_count)
col.find_one({"_id": "manual-lorca-2011"}, {"impacto": 1, "alerta": 1, "fuentes": 1, "modificado": 1})

Modificados: 1


{'_id': 'manual-lorca-2011',
 'fuentes': ['manual', 'ign'],
 'alerta': 'orange',
 'impacto': {'fallecidos': 9},
 'modificado': datetime.datetime(2026, 10, 24, 10, 51, 49, 456000)}

### Ejercicio 5 — Campo calculado con un pipeline de actualización
Añadimos a **todos** los documentos un campo `categoria` según la escala habitual de magnitudes:

| Magnitud | Categoría |
|---|---|
| < 6 | Moderado |
| 6 – 6.9 | Fuerte |
| 7 – 7.9 | Mayor |
| ≥ 8 | Gran terremoto |

In [6]:
resultado = col.update_many(
    {},
    [{"$set": {"categoria": {"$switch": {
        "branches": [
            {"case": {"$gte": ["$magnitud", 8]}, "then": "Gran terremoto"},
            {"case": {"$gte": ["$magnitud", 7]}, "then": "Mayor"},
            {"case": {"$gte": ["$magnitud", 6]}, "then": "Fuerte"},
        ],
        "default": "Moderado",
    }}}}],
)
print("Modificados:", resultado.modified_count)
for categoria in ["Moderado", "Fuerte", "Mayor", "Gran terremoto"]:
    print(f"{categoria:15} {col.count_documents({'categoria': categoria}):6}")

Modificados: 2
Moderado         17769
Fuerte            1303
Mayor              140
Gran terremoto       7


### Ejercicio 6 — Eliminar documentos (`delete_many`)
Borramos los documentos que insertamos a mano para dejar el dataset solo con datos de USGS.

In [7]:
resultado = col.delete_many({"fuentes": "manual"})
print("Eliminados:", resultado.deleted_count)
print("Total:", col.count_documents({}))

Eliminados: 2
Total: 19217


## 2.3 Proyección

### Ejercicio 7 — Los 10 terremotos más fuertes (proyección de inclusión)
Mostramos solo los campos relevantes y ocultamos el `_id`.

In [8]:
tabla(col.find(
    {},
    {"_id": 0, "magnitud": 1, "lugar": 1, "fecha": 1, "profundidad_km": 1, "tsunami": 1},
).sort("magnitud", -1).limit(10))

,magnitud,lugar,fecha,profundidad_km,tsunami
0,8.8,"2025 Kamchatka Peninsula, Russia Earthquake",2025-07-29 23:24:52.483,35.00,True
1,8.2,2018 Fiji Earthquake,2018-08-19 00:19:40.670,600.00,True
2,8.2,"2017 Tehuantepec, Mexico Earthquake",2017-09-08 04:49:19.180,47.39,True
3,8.2,"2021 Chignik, Alaska Earthquake",2021-07-29 06:15:49.188,35.00,True
4,8.1,"2021 Kermadec Islands, New Zealand Earthquake",2021-03-04 19:28:33.178,28.93,True
5,8.1,2021 South Sandwich Islands Earthquake,2021-08-12 18:35:17.231,22.79,False
6,8.0,"78 km NE of Navarro, Peru",2019-05-26 07:41:15.073,122.57,True
7,7.9,"45 km S of Levuka, Fiji",2018-09-06 15:49:18.710,670.81,True
8,7.9,"261 km SE of Chiniak, Alaska",2018-01-23 09:31:40.890,14.06,True
9,7.9,"35 km WNW of Panguna, Papua New Guinea",2017-01-22 04:30:22.960,135.00,True


### Ejercicio 8 — Campos de subdocumentos (notación punto)
Terremotos que sintieron más de 10.000 personas, mostrando campos que están dentro del subdocumento `impacto`.

In [9]:
tabla(col.find(
    {"impacto.personas_lo_sintieron": {"$gt": 10000}},
    {"_id": 0, "lugar": 1, "magnitud": 1,
     "impacto.personas_lo_sintieron": 1, "impacto.intensidad_reportada": 1},
).sort("impacto.personas_lo_sintieron", -1))

,magnitud,lugar,impacto.personas_lo_sintieron,impacto.intensidad_reportada
0,5.10,"4 km SE of Sparta, North Carolina",66722,6.4
1,5.80,"14 km NW of Pawnee, Oklahoma",48764,7.9
2,5.21,"5 km S of Julian, CA",42638,6.1
3,7.10,Ridgecrest Earthquake Sequence,39888,8.7
4,6.40,Ridgecrest Earthquake Sequence,27543,7.9
5,5.06,"8 km NW of Prague, Oklahoma",26480,6.9
6,5.70,"5 km NNE of Magna, Utah",26376,6.7
7,6.00,"Antelope Valley, CA",25987,5.7
8,5.06,"15km ESE of Alum Rock, CA",24205,4.6
9,6.50,"Stanley, Idaho",23163,7.6


### Ejercicio 9 — Proyección de exclusión y `$slice` sobre un array
Del terremoto de Kamchatka 2025 excluimos los subdocumentos `calidad` e `impacto` y mostramos solo los 5 primeros productos científicos.

In [10]:
col.find_one(
    {"_id": "us6000qw60"},
    {"calidad": 0, "impacto": 0, "url": 0, "productos": {"$slice": 5}},
)

{'_id': 'us6000qw60',
 'magnitud': 8.8,
 'tipo_magnitud': 'mww',
 'lugar': '2025 Kamchatka Peninsula, Russia Earthquake',
 'region': 'Russia',
 'fecha': datetime.datetime(2025, 7, 29, 23, 24, 52, 483000),
 'ubicacion': {'type': 'Point', 'coordinates': [160.2395, 52.4948]},
 'profundidad_km': 35,
 'tipo_evento': 'earthquake',
 'tsunami': True,
 'alerta': 'red',
 'significancia': 2114,
 'estado': 'reviewed',
 'red': 'us',
 'fuentes': ['at', 'us', 'pt', 'usauto'],
 'productos': ['dyfi',
  'earthquake-name',
  'event-sequence',
  'finite-fault',
  'general-link'],
 'categoria': 'Gran terremoto'}

## 2.4 Filtrado

### Ejercicio 10 — Rango de fechas, comparación y booleano
Terremotos de magnitud ≥ 7 que generaron alerta de tsunami desde 2024.

In [7]:
filtro = {
    "fecha": {"$gte": datetime(2024, 1, 1)},
    "magnitud": {"$gte": 7},
    "tsunami": True,
}
print(col.count_documents(filtro), "terremotos")
tabla(col.find(filtro, {"_id": 0, "fecha": 1, "magnitud": 1, "lugar": 1, "alerta": 1}).sort("fecha", 1))

24 terremotos


,magnitud,lugar,fecha,alerta
0,7.5,"2024 Noto Peninsula, Japan Earthquake",2024-01-01 07:10:09.476,red
1,7.4,"15 km S of Hualien City, Taiwan",2024-04-02 23:58:12.173,yellow
2,7.2,"10 km WSW of Atiquipa, Peru",2024-06-28 05:36:36.902,yellow
3,7.1,"106 km WSW of Sangay, Philippines",2024-07-11 02:13:18.585,green
4,7.4,"41 km ESE of San Pedro de Atacama, Chile",2024-07-19 01:50:48.571,green
5,7.0,"102 km E of Petropavlovsk-Kamchatsky, Russia",2024-08-17 19:10:26.820,green
6,7.0,"2024 Offshore Cape Mendocino, California Earthquake",2024-12-05 18:44:21.110,yellow
7,7.3,"24 km WNW of Port-Vila, Vanuatu",2024-12-17 01:47:25.741,yellow
8,7.6,"210 km SSW of George Town, Cayman Islands",2025-02-08 23:23:14.697,green
9,7.0,"61 km SSE of Pangai, Tonga",2025-03-30 12:18:50.362,green


### Ejercicio 11 — `$in`: alertas naranja y roja
El sistema PAGER de USGS asigna una alerta según las víctimas y pérdidas económicas estimadas. Buscamos los terremotos con alerta **roja** o **naranja**.

In [12]:
for color in ["red", "orange", "yellow", "green"]:
    print(f"{color:7} {col.count_documents({'alerta': color}):5}")

tabla(col.find(
    {"alerta": {"$in": ["red", "orange"]}, "fecha": {"$gte": datetime(2023, 1, 1)}},
    {"_id": 0, "fecha": 1, "alerta": 1, "magnitud": 1, "lugar": 1},
).sort("fecha", 1))

red        38


orange     52
yellow    234
green    6487


,magnitud,lugar,fecha,alerta
0,7.8,"Pazarcik earthquake, Kahramanmaras earthquake sequence",2023-02-06 01:17:34.342,red
1,7.5,"Elbistan earthquake, Kahramanmaras earthquake sequence",2023-02-06 10:24:48.811,red
2,6.3,"2 km NNW of Uzunba?, Turkey",2023-02-20 17:04:29.747,red
3,6.8,"14 km NNW of Baláo, Ecuador",2023-03-18 17:12:52.481,orange
4,5.6,"13 km SW of Khowy, Iran",2023-03-24 03:16:56.125,red
5,5.4,"24 km S of Dezhou, China",2023-08-05 18:33:59.457,red
6,5.6,"3 km E of Restrepo, Colombia",2023-08-17 17:17:17.698,orange
7,6.8,"Al Haouz, Morocco",2023-09-08 22:11:01.405,red
8,6.3,"32 km NNE of Zindah Jān, Afghanistan",2023-10-07 06:41:03.327,orange
9,6.3,"25 km NNE of Zindah Jān, Afghanistan",2023-10-07 07:12:49.702,orange


### Ejercicio 12 — `$exists`: campos que no están en todos los documentos
En el dataset los campos sin dato no aparecen en el documento. Contamos cuántos terremotos tienen o no información de impacto y de alerta.

In [13]:
print("Con impacto:      ", col.count_documents({"impacto": {"$exists": True}}))
print("Sin impacto:      ", col.count_documents({"impacto": {"$exists": False}}))
print("Con alerta PAGER: ", col.count_documents({"alerta": {"$exists": True}}))
print("Sin alerta, pero sentido por la población:",
      col.count_documents({"alerta": {"$exists": False}, "impacto.personas_lo_sintieron": {"$exists": True}}))

Con impacto:       10936


Sin impacto:       8281
Con alerta PAGER:  6811


Sin alerta, pero sentido por la población: 3998


### Ejercicio 13 — Consultas sobre arrays: `$all` y `$size`
- `$all`: terremotos para los que USGS publicó **a la vez** un modelo de falla finita (`finite-fault`) y una estimación de pérdidas (`losspager`).
- `$size`: terremotos registrados por una sola red sísmica frente a los registrados por cuatro.

In [14]:
print("Con finite-fault y losspager:", col.count_documents({"productos": {"$all": ["finite-fault", "losspager"]}}))
print("Con 1 fuente: ", col.count_documents({"fuentes": {"$size": 1}}))
print("Con 4 fuentes:", col.count_documents({"fuentes": {"$size": 4}}))

tabla(col.find(
    {"productos": {"$all": ["finite-fault", "losspager"]}, "magnitud": {"$gte": 7.8}},
    {"_id": 0, "magnitud": 1, "lugar": 1, "fecha": 1},
).sort("magnitud", -1))

Con finite-fault y losspager: 133
Con 1 fuente:  7269
Con 4 fuentes: 1182


,magnitud,lugar,fecha
0,8.8,"2025 Kamchatka Peninsula, Russia Earthquake",2025-07-29 23:24:52.483
1,8.2,"2017 Tehuantepec, Mexico Earthquake",2017-09-08 04:49:19.180
2,8.2,2018 Fiji Earthquake,2018-08-19 00:19:40.670
3,8.2,"2021 Chignik, Alaska Earthquake",2021-07-29 06:15:49.188
4,8.1,"2021 Kermadec Islands, New Zealand Earthquake",2021-03-04 19:28:33.178
5,8.0,"78 km NE of Navarro, Peru",2019-05-26 07:41:15.073
6,7.9,"140 km E of Kokopo, Papua New Guinea",2016-12-17 10:51:10.500
7,7.9,"35 km WNW of Panguna, Papua New Guinea",2017-01-22 04:30:22.960
8,7.9,"261 km SE of Chiniak, Alaska",2018-01-23 09:31:40.890
9,7.8,"southwest of Sumatra, Indonesia",2016-03-02 12:49:48.110


### Ejercicio 14 — Operadores lógicos `$or` / `$and` y expresiones regulares
Terremotos **profundos** (más de 300 km) en Sudamérica: la región es Chile, Perú o Argentina, **o** el lugar contiene la palabra “Bolivia”.

In [15]:
filtro = {
    "$and": [
        {"profundidad_km": {"$gt": 300}},
        {"$or": [
            {"region": {"$in": ["Chile", "Peru", "Argentina"]}},
            {"lugar": {"$regex": "bolivia", "$options": "i"}},
        ]},
    ]
}
print(col.count_documents(filtro), "terremotos profundos en Sudamérica")
tabla(col.find(filtro, {"_id": 0, "magnitud": 1, "profundidad_km": 1, "lugar": 1})
      .sort("profundidad_km", -1).limit(10))

34 terremotos profundos en Sudamérica


,magnitud,lugar,profundidad_km
0,5.6,"126 km WNW of Iñapari, Peru",648.298
1,7.1,"136 km W of Iñapari, Peru",630.000
2,6.4,"186 km NW of Iñapari, Peru",612.000
3,5.3,"34 km N of Monteagudo, Bolivia",599.020
4,5.2,"30 km SW of Colonia Dora, Argentina",597.130
5,6.8,"29 km SW of Campo Gallo, Argentina",596.837
6,5.1,"38 km NW of Colonia Dora, Argentina",596.070
7,6.5,"41 km E of Padilla, Bolivia",595.980
8,5.6,"37 km NW of Añatuya, Argentina",593.431
9,5.0,"39 km NW of Añatuya, Argentina",589.620


### Ejercicio 15 — `$ne` y `$nin`: eventos que no son terremotos
No todo lo que registra un sismógrafo es un terremoto. Con `$ne` contamos los eventos de otro tipo y con `$nin` excluimos
también las erupciones volcánicas (casi todas del Kilauea, Hawái, 2018). Aparecen las **pruebas nucleares de Corea del Norte**.

In [16]:
for tipo in col.distinct("tipo_evento"):
    print(f"{tipo:18} {col.count_documents({'tipo_evento': tipo}):6}")
print("No son terremotos:", col.count_documents({"tipo_evento": {"$ne": "earthquake"}}))

tabla(col.find(
    {"tipo_evento": {"$nin": ["earthquake", "volcanic eruption"]}},
    {"_id": 0, "tipo_evento": 1, "fecha": 1, "magnitud": 1, "lugar": 1},
).sort("fecha", 1))

earthquake          19157
landslide               2
nuclear explosion       3


volcanic eruption      55
No son terremotos: 60


,magnitud,lugar,fecha,tipo_evento
0,5.1,"21 km ENE of S?ngjibaegam, North Korea",2016-01-06 01:30:01.480,nuclear explosion
1,5.3,"23 km ENE of S?ngjibaegam, North Korea",2016-09-09 00:30:01.440,nuclear explosion
2,6.3,"21 km ENE of S?ngjibaegam, North Korea",2017-09-03 03:30:01.760,nuclear explosion
3,5.4,"44 km NNE of Hobart Bay, Alaska",2025-08-10 13:26:43.390,landslide
4,5.2,"55 km NW of Kodāri̇̄, Nepal",2026-08-26 02:52:10.000,landslide


### Ejercicio 16 — Filtro geoespacial (`$near`)
Creamos un índice `2dsphere` sobre `ubicacion` y buscamos los terremotos a menos de **800 km de Madrid**, del más cercano al más lejano.

In [17]:
col.create_index([("ubicacion", GEOSPHERE)])

madrid = {"type": "Point", "coordinates": [-3.7038, 40.4168]}
tabla(col.find(
    {"ubicacion": {"$near": {"$geometry": madrid, "$maxDistance": 800_000}}},
    {"_id": 0, "fecha": 1, "magnitud": 1, "lugar": 1},
))

,magnitud,lugar,fecha
0,5.0,"1 km W of Alhendín, Spain",2026-08-14 23:04:40.798
1,5.2,"23 km E of San Jos?, Spain",2025-07-14 05:13:28.590
2,5.1,"66 km NE of Al Hoceïma, Morocco",2016-02-22 03:46:02.460
3,5.0,"58 km NNE of Al Hoceïma, Morocco",2016-03-11 04:16:47.910
4,5.2,"55 km NNE of Al Hoceïma, Morocco",2016-03-15 04:40:39.110
5,5.3,"51 km NNE of Al Hoceïma, Morocco",2016-01-25 04:34:19.170
6,6.3,"49 km NNE of Al Hoceïma, Morocco",2016-01-25 04:22:02.730
7,5.1,"48 km NNE of Al Hoceïma, Morocco",2016-01-21 13:47:19.860
8,5.0,"39 km NNE of Al Hoceïma, Morocco",2016-01-25 04:30:55.190
9,5.5,Strait of Gibraltar,2022-05-20 12:35:49.576
